# Copy final figures and tables next to the LaTeX sources

Reads every compilable `.tex` in `1-paper/` and `2-appendix/`, finds each `\includegraphics{graphs/...}`, `\input{tables/...}` and `\inputbody{tables/...}`, looks the file up in `replication-package/graphs|tables/(paper|appendix)/`, and copies it to the flat `graphs/` or `tables/` folder next to that `.tex` (e.g. `1-paper/graphs/Figure-1-a.pdf`, `2-appendix/tables/Table-D3.tex`). Only the files each document uses are copied. Nothing needs to be configured: the LaTeX files decide what is copied.

In [ ]:
import glob
import os
import re
import shutil

PKG = "../../"        # replication-package/ (this notebook runs from progs/Python/)
ROOT = "../../../"    # folder containing 1-paper/ and 2-appendix/
DOCS = ["1-paper", "2-appendix"]
GRAPHIC_EXTS = [".pdf", ".png", ".jpg", ".jpeg", ".eps", ".svg"]

In [ ]:
def references(tex_file):
    """(kind, path) for every graphic and table input in a .tex file (comments ignored)."""
    with open(tex_file, encoding="utf-8", errors="replace") as fh:
        text = re.sub(r"(?<!\\)%[^\n]*", "", fh.read())
    refs = [("graphic", m) for m in re.findall(r"\\includegraphics\s*(?:\[[^\]]*\])?\s*\{([^}]+)\}", text)]
    refs += [("table", m) for m in re.findall(r"\\input(?:body)?\s*\{([^}]+)\}", text)]
    return [(k, re.sub(r"^\./", "", p.strip())) for k, p in refs]


def package_files(ref, kind):
    """Files in replication-package/<graphs|tables>/(paper|appendix)/ for a reference graphs/<name>[.ext]."""
    exts = GRAPHIC_EXTS if kind == "graphic" else [".tex"]
    folder, name = ref.split("/", 1)
    hits = []
    for loc in ("paper", "appendix"):
        p = os.path.join(PKG, folder, loc, name)
        if os.path.splitext(p)[1].lower() in exts:
            hits += [p] if os.path.exists(p) else []
        else:
            hits += [f for f in glob.glob(glob.escape(p) + ".*") if os.path.splitext(f)[1].lower() in exts]
    return hits

In [ ]:
copied, missing = 0, []
for doc in DOCS:
    for tex in glob.glob(os.path.join(ROOT, doc, "*.tex")):
        with open(tex, encoding="utf-8", errors="replace") as fh:
            if "\\documentclass" not in fh.read():
                continue
        for kind, ref in references(tex):
            if not re.match(r"(graphs|tables)/[^/]+$", ref):
                continue
            files = package_files(ref, kind)
            if not files:
                missing.append(f"{doc}: {ref}")
            for f in files:
                dest = os.path.join(ROOT, doc, ref.split("/")[0], os.path.basename(f))
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copyfile(f, dest)
                copied += 1
print(f"Copied {copied} file(s)")
if missing:
    print("MISSING (not found in replication-package/graphs|tables/(paper|appendix)/):")
    for m in missing:
        print("  ", m)